<img src="../assets/rpt_risk_overview.svg" alt="SAP RPT 신용위험 분류 흐름" style="display:block;width:100%;max-width:1200px;margin:0 auto 24px;border-radius:12px;">


# 03 · SAP RPT 신용위험 분류

**목표**: ID를 제거한 교육용 표 데이터를 SAP RPT-1.5 Large로 80/20 holdout 평가하고, 선택 시 BTP Object Storage·AI Core Artifact에 저장한 뒤 가명 holdout 행 한 건을 재예측해 사람 검토용으로 확인합니다.

기본 평가는 읽기 전용 prediction입니다. `BTP에 시나리오 저장` 버튼만 `ALLOW_WRITE_TOOLS=true`와 Object Storage 설정이 있을 때 교육용 holdout snapshot을 저장하고 AI Core `tabular-orchestration` Artifact로 등록합니다. BTP 학습 실행·deployment 생성·삭제와 자동 신용 승인·거절은 범위에 포함하지 않습니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. RPT 분류 UI</h2>
<p style="margin: 7px 0;">UI는 정제 데이터 상태를 확인한 뒤 80:20 holdout을 평가합니다. 평가 완료 후에는 교육용 snapshot을 BTP에 명시적으로 저장하고, 저장된 holdout 행을 한 건씩 같은 support set으로 재예측할 수 있습니다. RPT와 저장 snapshot에는 feature·가명 record ID·label만 사용하며 원본 ID 및 감사 속성은 전송·저장하지 않습니다.</p>
</div>
</div>

In [9]:
from pathlib import Path
import importlib
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').is_dir())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '3_gen_ai_hub_services')]
import src.rpt_risk.lab_core as rpt_risk_lab_core
import src.rpt_risk.ui as rpt_risk_ui

importlib.reload(rpt_risk_lab_core)
rpt_risk_ui = importlib.reload(rpt_risk_ui)
rpt_risk_app = rpt_risk_ui.build_rpt_risk_ui()
rpt_risk_app.launch(inline=True, share=False, show_error=True, server_name='127.0.0.1', height=900)

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ 체크포인트</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">데이터와 의사결정의 경계</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>원본 customer ID와 중복 문자열 위험 라벨은 교육용 CSV에 보관하지 않습니다.</li><li>GENDER와 MARITAL은 RPT 입력이 아니라 로컬 감사 집계용으로만 사용합니다.</li><li>분류 label과 confidence는 자동 승인·거절이 아닌 사람 검토용 참고치입니다.</li></ul>
</div>
</div>
<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 정제 데이터와 전송 경계 확인</h2>
<p style="margin: 7px 0;">public API는 행 수, 0/1 타깃 분포, RPT feature, 제외 열과 교육용 원본 행 미리보기를 반환합니다. 이 단계는 BTP를 호출하지 않으며 customer ID는 표에 포함하지 않습니다.</p>
</div>
</div>

In [5]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').is_dir())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '3_gen_ai_hub_services')]
from src.rpt_risk.lesson_support.config import load_settings
from src.rpt_risk.lab_core import explain_rpt_risk_case, inspect_risk_dataset, predict_rpt_risk_case, preview_risk_dataset, run_rpt_permutation_importance, run_rpt_risk_evaluation

settings = load_settings()
dataset_evidence = inspect_risk_dataset()
data_preview = preview_risk_dataset(limit=20)
import pandas as pd
display(pd.DataFrame(data_preview['rows']))
dataset_evidence

,record_id,age,income,num_kids,num_cards,how_paid,employed_days,store_car,loans,mortgage,mortgage_amount,credit_score,audit_gender,audit_marital,risk_label
0,risk-00001,44,59944,1,2,monthly,6994,2,0,y,134000,0.76,m,married,1
1,risk-00002,35,59692,1,1,monthly,3750,1,0,y,193000,0.62,m,married,0
2,risk-00003,34,59508,1,1,monthly,3198,2,1,y,77000,0.74,m,married,1
3,risk-00004,34,59463,0,2,monthly,2784,1,1,y,118000,0.33,m,married,0
4,risk-00005,39,59393,0,2,monthly,5510,1,0,y,59000,0.74,f,married,1
5,risk-00006,41,59276,1,2,monthly,6996,1,1,y,70000,0.79,m,married,1
6,risk-00007,42,59201,0,1,monthly,5610,2,0,y,59000,0.77,m,married,1
7,risk-00008,31,59193,1,2,monthly,2981,1,1,y,138000,0.75,f,married,1
8,risk-00009,28,59179,1,1,monthly,1589,2,1,y,193000,0.63,m,married,0
9,risk-00010,30,59036,1,1,monthly,3768,2,1,y,75000,0.66,m,married,1


{'dataset_path': '/Users/I781176/Desktop/devday2026/3_gen_ai_hub_services/data/risk_prova_educational.csv',
 'row_count': 4117,
 'feature_columns': ['age',
  'income',
  'num_kids',
  'num_cards',
  'how_paid',
  'employed_days',
  'store_car',
  'loans',
  'mortgage',
  'mortgage_amount',
  'credit_score'],
 'target_column': 'risk_label',
 'target_distribution': {'0': 3313, '1': 804},
 'rpt_excluded_columns': ['source_id',
  'risk_text_label',
  'audit_gender',
  'audit_marital',
  'blank_or_unreliable_source_columns'],
 'audit_only_columns': ['audit_gender', 'audit_marital'],
 'data_boundary': 'RPT에는 feature_columns와 risk_label만 전송합니다. 식별자와 감사 속성은 전송하지 않습니다.'}

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. 80/20 Holdout 평가</h2>
<p style="margin: 7px 0;">각 class를 같은 비율로 나눠 80%의 실제 label을 RPT 문맥으로 전달하고, 나머지 20%의 target에는 [PREDICT] placeholder를 둡니다. 함수가 지표·예측·감사 집계를 함께 반환합니다.</p>
</div>
</div>

In [6]:
evaluation = run_rpt_risk_evaluation(settings)
display(pd.DataFrame(evaluation['prediction_table']).head(20))
{
    'model': evaluation['model'],
    'request_summary': evaluation['request_summary'],
    'metrics': evaluation['metrics'],
    'review_notice': evaluation['review_notice'],
}

,record_id,age,income,num_kids,num_cards,how_paid,employed_days,store_car,loans,mortgage,mortgage_amount,credit_score,audit_gender,audit_marital,risk_label,rpt_prediction,rpt_confidence
0,risk-00009,28,59179,1,1,monthly,1589,2,1,y,193000,0.63,m,married,0,0,1.00
1,risk-00018,32,57718,1,2,monthly,3012,1,1,y,192000,0.43,m,married,0,0,1.00
2,risk-00029,38,56590,0,2,monthly,3675,1,0,y,113000,0.56,m,married,0,0,1.00
3,risk-00036,36,55777,1,2,monthly,4709,2,0,y,178000,0.51,f,married,0,0,1.00
4,risk-00037,38,55752,0,2,monthly,4335,2,1,y,111000,0.76,m,married,1,1,0.91
5,risk-00050,35,54673,1,2,monthly,3808,2,1,y,124000,0.46,f,married,0,0,1.00
6,risk-00055,33,54013,0,1,monthly,3190,2,0,y,54000,0.86,f,married,1,1,1.00
7,risk-00065,36,53365,0,1,monthly,3630,2,0,y,106000,0.51,m,married,0,0,1.00
8,risk-00066,42,53307,0,2,monthly,8088,1,1,y,159000,0.36,m,married,0,0,1.00
9,risk-00069,40,53180,1,2,monthly,6739,1,0,y,116000,0.69,f,married,1,1,0.77


{'model': 'sap-rpt-1.5-large',
 'request_summary': {'support_rows': 3293,
  'holdout_rows': 824,
  'rpt_request_count': 2,
  'max_query_rows_per_request': 512,
  'feature_columns': ['age',
   'income',
   'num_kids',
   'num_cards',
   'how_paid',
   'employed_days',
   'store_car',
   'loans',
   'mortgage',
   'mortgage_amount',
   'credit_score'],
  'target_column': 'risk_label'},
 'metrics': {'accuracy': 0.970873786407767,
  'precision_positive_1': 0.9536423841059603,
  'recall_positive_1': 0.8944099378881988,
  'f1_positive_1': 0.9230769230769231,
  'confusion_matrix': {'actual_0': {'predicted_0': 656, 'predicted_1': 7},
   'actual_1': {'predicted_0': 17, 'predicted_1': 144}}},
 'review_notice': '이 결과는 교육용 모델 평가이며 신용 승인·거절 결정을 대신하지 않습니다.'}

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. 감사용 집계와 단건 분류</h2>
<p style="margin: 7px 0;">audit은 holdout 정확도와 label 1 예측률을 그룹별로 표시할 뿐 공정성을 자동 판정하지 않습니다. 아래 가명 사례도 같은 public API로 예측합니다.</p>
</div>
</div>

In [7]:
sample_case = {
    'age': 35, 'income': 59692, 'num_kids': 1, 'num_cards': 1,
    'how_paid': 'monthly', 'employed_days': 3750, 'store_car': 1,
    'loans': 0, 'mortgage': 'y', 'mortgage_amount': 193000, 'credit_score': 0.62,
}
{
    'audit': evaluation['audit'],
    'single_case': predict_rpt_risk_case(sample_case, settings),
}

{'audit': {'audit_gender': [{'group': 'f',
    'count': 404,
    'accuracy': 0.9702970297029703,
    'predicted_positive_rate': 0.16584158415841585},
   {'group': 'm',
    'count': 420,
    'accuracy': 0.9714285714285714,
    'predicted_positive_rate': 0.2}],
  'audit_marital': [{'group': 'divsepwid',
    'count': 178,
    'accuracy': 0.9550561797752809,
    'predicted_positive_rate': 0.19101123595505617},
   {'group': 'married',
    'count': 396,
    'accuracy': 0.9797979797979798,
    'predicted_positive_rate': 0.20202020202020202},
   {'group': 'single',
    'count': 250,
    'accuracy': 0.968,
    'predicted_positive_rate': 0.148}]},
 'single_case': {'model': 'sap-rpt-1.5-large',
  'classification': '0',
  'confidence': 1.0,
  'request_summary': {'support_rows': 3293,
   'query_rows': 1,
   'feature_columns': ['age',
    'income',
    'num_kids',
    'num_cards',
    'how_paid',
    'employed_days',
    'store_car',
    'loans',
    'mortgage',
    'mortgage_amount',
    'credit_sc

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. 저장된 RPT 시나리오 로드와 단건 재예측</h2>
<p style="margin: 7px 0;">BTP에 저장된 Holdout Run 목록에서 최신 Artifact를 불러온 뒤, 저장된 20% holdout 행 한 건을 선택해 다시 예측합니다. 재예측 시에는 저장 당시의 80% support set을 사용하므로 같은 시나리오의 결과를 사람 검토용으로 재현할 수 있습니다.</p>
<p style="margin: 7px 0;"><strong>실행 전제:</strong> 먼저 UI에서 80:20 평가를 실행하고 <code>BTP에 시나리오 저장</code>을 완료해야 합니다. 이 코드 셀은 BTP에 새 데이터를 쓰지 않고, 저장된 Artifact를 읽어 예측만 수행합니다.</p>
</div>
</div>

In [8]:
# 5. 저장된 RPT Holdout 시나리오 로드 → 단건 재예측
from src.rpt_risk.lab_core import (
    list_saved_rpt_holdout_runs,
    load_saved_rpt_holdout_run,
    predict_saved_rpt_holdout_row,
    saved_rpt_holdout_scenarios,
)

saved_runs = list_saved_rpt_holdout_runs(settings)['runs']
if not saved_runs:
    raise RuntimeError('저장된 Holdout Run이 없습니다. UI에서 80:20 평가 후 BTP에 시나리오 저장을 먼저 실행하세요.')

display(pd.DataFrame(saved_runs))
selected_run = saved_runs[0]  # 최신 Run. 다른 Run을 사용하려면 이 값을 바꾸세요.
saved_snapshot = load_saved_rpt_holdout_run(selected_run['artifact_id'], settings)
saved_rows = saved_rpt_holdout_scenarios(saved_snapshot)
selected_record_id = saved_rows[0]['record_id']  # 다른 holdout 행으로 바꿀 수 있습니다.
single_saved_prediction = predict_saved_rpt_holdout_row(saved_snapshot, selected_record_id, settings)

{
    'artifact_id': selected_run['artifact_id'],
    'stored_run': selected_run['name'],
    'selected_holdout_features': next(row for row in saved_rows if row['record_id'] == selected_record_id),
    'single_row_prediction': single_saved_prediction,
}


,artifact_id,name,object_uri,created_at
0,8ad67b41-03ca-433a-8a48-179b0bf98471,rpt-risk-holdout-20261105T092207600777+0900-cd...,s3://hcp-3943d145-8abb-4d1b-8deb-c035e64abd1c/...,2026-11-05T00:22:11+00:00


{'artifact_id': '8ad67b41-03ca-433a-8a48-179b0bf98471',
 'stored_run': 'rpt-risk-holdout-20261105T092207600777+0900-cd485743',
 'selected_holdout_features': {'record_id': 'risk-00009',
  'age': 28,
  'income': 59179,
  'num_kids': 1,
  'num_cards': 1,
  'how_paid': 'monthly',
  'employed_days': 1589,
  'store_car': 2,
  'loans': 1,
  'mortgage': 'y',
  'mortgage_amount': 193000,
  'credit_score': 0.63},
 'single_row_prediction': {'artifact_id': '8ad67b41-03ca-433a-8a48-179b0bf98471',
  'record_id': 'risk-00009',
  'classification': '0',
  'confidence': 1.0,
  'actual_label': '0',
  'correct': True,
  'stored_model': 'sap-rpt-1.5-large',
  'active_model': 'sap-rpt-1.5-large',
  'request_summary': {'support_rows': 3293,
   'query_rows': 1,
   'feature_columns': ['age',
    'income',
    'num_kids',
    'num_cards',
    'how_paid',
    'employed_days',
    'store_car',
    'loans',
    'mortgage',
    'mortgage_amount',
    'credit_score']},
  'execution_boundary': '저장된 교육용 holdout 행을 RPT